# Bus Type Assignment in a Synthetic Grid

In this notebook, we demonstrate how to assign bus types to the nodes in a synthetically generated grid.

There exists a **correlation between the bus types and the grid topology characteristics**, which was investigated by Elyas et al. (2016). The authors proposed a numerical measure, namely **bus type entropy**, to characterize this correlation and then a statistics-based method to search the best bus type assignments.

In [ ]:
# Install powergrid_synth from the GitHub repository
# Note: pip may report dependency conflicts with pre-installed Colab packages
# (google-colab, tensorflow). These warnings are harmless and can be ignored —
# powergrid_synth does not use those packages.
!pip install -q git+https://github.com/cookbook-ms/Chung_Lu_Chain-synthesizer.git

# Restart the runtime so that the upgraded numpy/scipy are loaded properly.
# After restart, run the cells below this one (this cell does NOT need to be re-run).
import os
os.kill(os.getpid(), 9)

## Basics

In [ ]:
import networkx as nx
from collections import Counter

from powergrid_synth.generator import PowerGridGenerator
from powergrid_synth.input_configurator import InputConfigurator
from powergrid_synth.bus_type_allocator import BusTypeAllocator
from powergrid_synth.visualization import GridVisualizer

## Synthetic raw topology generation

First, we generate a raw grid topology using the CLC model (see the [TopologyGeneration](TopologyGeneration_colab.ipynb) notebook for details).

In [ ]:
# Define 3 voltage levels
level_specs = [
    {'n': 20, 'avg_k': 4.0, 'diam': 6, 'dist_type': 'dgln'},
    {'n': 20, 'avg_k': 3.0, 'diam': 10, 'dist_type': 'dpl'},
    {'n': 10, 'avg_k': 2.0, 'diam': 10, 'dist_type': 'dgln'}
]

connection_specs = {
    (0, 1): {'type': 'k-stars', 'c': 0.174, 'gamma': 4.15},
    (1, 2): {'type': 'k-stars', 'c': 0.15, 'gamma': 4.15}
}

configurator = InputConfigurator(seed=100)
params = configurator.create_params(level_specs, connection_specs)

gen = PowerGridGenerator(seed=100)
grid_graph = gen.generate_grid(
    degrees_by_level=params['degrees_by_level'],
    diameters_by_level=params['diameters_by_level'],
    transformer_degrees=params['transformer_degrees'],
    keep_lcc=True,
)
print(f"Grid Generated: {grid_graph.number_of_nodes()} nodes, {grid_graph.number_of_edges()} edges")

## Bus Type Assignment

In [ ]:
allocator = BusTypeAllocator(grid_graph, entropy_model=1)
# We use a moderate iteration count for the demo
bus_types = allocator.allocate(max_iter=100, population_size=100)

In [ ]:
allocator.plot_entropy_pdf(figsize=(7,4))

In [ ]:
# Assign bus types as graph node attributes
nx.set_node_attributes(grid_graph, bus_types, name="bus_type")
# Show stats
counts = Counter(bus_types.values())
total = sum(counts.values())
print(f"-----> Assignment Complete:")
print(f"       Generators: {counts['Gen']} ({counts['Gen']/total:.1%})")
print(f"       Loads:      {counts['Load']} ({counts['Load']/total:.1%})")
print(f"       Connectors: {counts['Conn']} ({counts['Conn']/total:.1%})")

# --- Bus Type Visualization ---
print("\n[5] Visualizing Bus Types & Edge Styles...")

viz = GridVisualizer()
viz.plot_bus_types(
    grid_graph, 
    layout='kamada_kawai', 
    title="Bus Types & Transmission Lines", 
    figsize=(7,5)
)